In [1]:
"""
=============================================================================
TRIPLE-FILTERED PEAD STRATEGY — COMPREHENSIVE STATS SCRIPT
=============================================================================
Replicates every stat and figure referenced in the PEAD Strategy Primer PDF,
adapted to the Phase-9 (qqq-parking, open-price entry/exit) backtest engine
from pead.ipynb.

Key methodological features:
  • Expanding-window SUE quintile ranking (no look-ahead)
  • After-close announcements: entry shifted to next business day OPEN
  • Point-in-time S&P 500 membership filter (survivorship-bias free)
  • qqq parking: uninvested capital held in qqq at open prices
  • Entry at open, exit at open after 22 trading days (≈21-day hold)
  • 16 bps one-way fee (commission + slippage)
  • Max 7 simultaneous positions
  • Sectors: Industrials & Technology only

Outputs every stat block from the primer:
  1. Signal-level statistics (SUE horizon analysis, CAR distributions)
  2. Strategy returns (CAGR, cumulative, dollar values, DCA)
  3. Risk metrics (Sharpe, Sortino, Calmar, VaR, CVaR)
  4. Statistical significance (NW HAC t-test, CAPM, FF3/FF5/FF6, Lo Sharpe)
  5. Monte Carlo simulation (10,000 permutations)
  6. Outlier fragility (remove top 1/2/5/10%)
  7. Fee sensitivity (5 → 100 bps)
  8. Crisis / regime performance (9 market events + 4 eras)
  9. Alpha decay analysis (rolling Jensen's alpha, Mann-Kendall, Chow break)
 10. Benchmark comparison (qqq, QQQ, 70/30 QQQ/XLI)
 11. Trade-level statistics
 12. Year-by-year table

Usage:
    python pead_full_stats.py

Requirements:
    pip install pandas numpy scipy statsmodels yfinance tqdm pymannkendall
=============================================================================
"""

import sqlite3, warnings, sys, os
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
import yfinance as yf
from tqdm import tqdm

warnings.filterwarnings("ignore")

# ──────────────────────────────────────────────────────────────────────────────
# CONFIGURATION  —  edit these paths / parameters as needed
# ──────────────────────────────────────────────────────────────────────────────
PEAD_DB      = "pead_clean.db"       # earnings signal database
PIT_DB       = "real_backtest.db"    # point-in-time S&P 500 membership
SIM_START    = pd.Timestamp("2013-01-01")
SIM_END      = pd.Timestamp("2025-12-31")
CAPITAL      = 10_000
N_POSITIONS  = 7
FEE_RT       = 0.0016                # 16 bps per side
ROT_THRESH   = 0.5                   # SUE rotation threshold
MIN_HOLD     = 5                     # minimum hold days before rotation
RF           = 0.02                  # annual risk-free rate
TARGET_SECTORS = {"Industrials", "Technology"}

# Fama-French factor file (Kenneth French library CSV)
# If not present we skip FF regressions gracefully.
FF_CSV       = "F-F_Research_Data_5_Factors_2x3_daily.csv"
MOM_CSV      = "F-F_Momentum_Factor_daily.csv"

SEP  = "=" * 62
SEP2 = "-" * 58

# ──────────────────────────────────────────────────────────────────────────────
# COLORS (for any optional matplotlib output)
# ──────────────────────────────────────────────────────────────────────────────
GREEN  = "#22C55E"
BLUE   = "#3B82F6"
RED    = "#EF4444"
GOLD   = "#F59E0B"
PURPLE = "#8B5CF6"


# =============================================================================
# SECTION 0 — HELPERS
# =============================================================================
def cagr(eq):
    eq = eq.dropna()
    yrs = len(eq) / 252
    if yrs <= 0 or eq.iloc[0] <= 0:
        return 0.0
    return (eq.iloc[-1] / eq.iloc[0]) ** (1 / yrs) - 1


def max_drawdown(eq):
    return (eq / eq.cummax() - 1).min()


def sharpe(eq, rf=RF):
    dr = eq.pct_change(fill_method=None).dropna()
    rf_d = rf / 252
    exc = dr - rf_d
    return exc.mean() / exc.std() * np.sqrt(252) if exc.std() != 0 else 0


def sortino(eq, rf=RF):
    dr = eq.pct_change(fill_method=None).dropna()
    rf_d = rf / 252
    exc = dr - rf_d
    down = exc[exc < 0].std() * np.sqrt(252)
    return exc.mean() * np.sqrt(252) / down if down != 0 else 0


def calmar(eq, rf=RF):
    c = cagr(eq)
    mdd = abs(max_drawdown(eq))
    return c / mdd if mdd != 0 else 0


def omega(eq, threshold=0.0):
    dr = eq.pct_change(fill_method=None).dropna()
    g = dr[dr > threshold] - threshold
    l = threshold - dr[dr <= threshold]
    return g.sum() / l.sum() if l.sum() != 0 else np.inf


def perf(eq, rf=RF):
    eq = eq.dropna()
    eq = eq[eq > 0]
    if len(eq) < 2:
        return {}
    dr = eq.pct_change(fill_method=None).dropna()
    return dict(
        cagr=cagr(eq), vol=dr.std()*np.sqrt(252),
        mdd=max_drawdown(eq), sharpe=sharpe(eq, rf),
        sortino=sortino(eq, rf), calmar=calmar(eq, rf),
        cum=eq.iloc[-1]/eq.iloc[0]-1,
        skew=float(dr.skew()), kurt=float(dr.kurtosis()),
        var95=float(np.percentile(dr, 5)),
        cvar95=float(dr[dr <= np.percentile(dr, 5)].mean()),
        var99=float(np.percentile(dr, 1)),
        cvar99=float(dr[dr <= np.percentile(dr, 1)].mean()),
    )


def get_px(df, ticker, date, col="Open"):
    if ticker not in df.columns:
        return None
    for offset in range(4):
        d = date + pd.offsets.BDay(offset)
        try:
            v = df.loc[d, ticker]
            if pd.notna(v) and v > 0:
                return float(v)
        except KeyError:
            pass
    return None


def build_bench(px_close, tkr_wts, index, start_date):
    curves = []
    for tkr, wt in tkr_wts.items():
        raw = px_close[tkr].copy() if tkr in px_close.columns else None
        if raw is None:
            continue
        if raw.index.tz is not None:
            raw.index = raw.index.tz_localize(None)
        aligned = raw.reindex(index.union(raw.index)).ffill().reindex(index)
        base_val = aligned.loc[start_date] if start_date in aligned.index else aligned.dropna().iloc[0]
        curves.append(aligned / base_val * wt)
    return sum(curves) if curves else pd.Series(dtype=float)


def hurst_exponent(ts, max_lag=100):
    """R/S rescaled range Hurst exponent."""
    ts = np.array(ts, dtype=float)
    ts = ts[~np.isnan(ts)]
    lags = range(2, min(max_lag, len(ts) // 2))
    rs_vals = []
    for lag in lags:
        chunks = [ts[i:i+lag] for i in range(0, len(ts)-lag, lag)]
        rs_chunk = []
        for c in chunks:
            m = np.mean(c)
            z = np.cumsum(c - m)
            r = np.max(z) - np.min(z)
            s = np.std(c, ddof=1)
            if s > 0:
                rs_chunk.append(r / s)
        if rs_chunk:
            rs_vals.append((np.log(lag), np.log(np.mean(rs_chunk))))
    if len(rs_vals) < 5:
        return 0.5
    xs, ys = zip(*rs_vals)
    return np.polyfit(xs, ys, 1)[0]


def mann_kendall(x):
    """Simple Mann-Kendall trend test. Returns (trend_str, s, var_s, z, p)."""
    n = len(x)
    s = sum(np.sign(x[j] - x[i]) for i in range(n-1) for j in range(i+1, n))
    var_s = n*(n-1)*(2*n+5) / 18
    z = (s - 1) / np.sqrt(var_s) if s > 0 else (s + 1) / np.sqrt(var_s) if s < 0 else 0
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    trend = "increasing" if z > 0 else "decreasing"
    return trend, s, var_s, z, p


# =============================================================================
# SECTION 1 — LOAD & CLEAN SIGNAL DATABASE
# =============================================================================
print(SEP)
print("  STEP 1 — LOADING SIGNAL DATABASE")
print(SEP)

if not os.path.exists(PEAD_DB):
    sys.exit(f"ERROR: {PEAD_DB} not found. Place it in the same directory as this script.")

conn   = sqlite3.connect(PEAD_DB)
df_raw = pd.read_sql("SELECT * FROM pead_clean", conn)
conn.close()

df_raw["Date_ts"]    = pd.to_datetime(df_raw["Date"], utc=True, errors="coerce")
df_raw["Date_naive"] = (pd.to_datetime(df_raw["Date_naive"], utc=True, errors="coerce")
                          .dt.normalize().dt.tz_localize(None))
df_raw["hour_utc"]   = df_raw["Date_ts"].dt.hour
df_raw["Quarter"]    = df_raw["Quarter"].apply(lambda x: pd.Period(x, freq="Q"))

# After-close entry shift (>= 20:00 UTC = after 4 pm EST)
df_raw["entry_date"] = np.where(
    df_raw["hour_utc"] >= 20,
    df_raw["Date_naive"].apply(lambda d: pd.bdate_range(start=d, periods=2)[-1]),
    df_raw["Date_naive"],
)
df_raw["entry_date"] = pd.to_datetime(df_raw["entry_date"])

after_close_pct = (df_raw["hour_utc"] >= 20).mean()
print(f"  Loaded rows        : {len(df_raw):,}")
print(f"  Unique tickers     : {df_raw['Ticker'].nunique():,}")
print(f"  Entry shifted +1BD : {after_close_pct*100:.1f}% of signals (after-close fix)")

# Deduplicate (same ticker, same entry date → keep highest SUE)
dupes = df_raw.duplicated(subset=["Ticker", "entry_date"]).sum()
if dupes:
    df_raw = (df_raw.sort_values("SUE", ascending=False)
              .drop_duplicates(subset=["Ticker", "entry_date"])
              .sort_values("entry_date")
              .reset_index(drop=True))
    print(f"  Deduplicated       : {dupes:,} rows removed")

df_raw = df_raw[df_raw["SUE"].notna()].copy()

# =============================================================================
# SECTION 2 — EXPANDING-WINDOW SUE QUINTILE (no look-ahead)
# =============================================================================
print(f"\n  Building SUE_Q_clean (expanding window — no look-ahead)...")
print(f"  This may take 60–120 seconds...")

df_raw = df_raw.sort_values(["Quarter", "entry_date"]).reset_index(drop=True)
df_raw["SUE_Q_clean"] = np.nan

quarters = df_raw["Quarter"].unique()
for quarter in tqdm(quarters, desc="  Quarters", ncols=70):
    q_mask = df_raw["Quarter"] == quarter
    q_grp  = df_raw.loc[q_mask]
    for ann_date in sorted(q_grp["entry_date"].unique()):
        known_mask = q_mask & (df_raw["entry_date"] <= ann_date)
        known_sues = df_raw.loc[known_mask, "SUE"].dropna()
        if len(known_sues) < 5:
            continue
        today_mask = q_mask & (df_raw["entry_date"] == ann_date) & df_raw["SUE"].notna()
        b = np.percentile(known_sues.values, [20, 40, 60, 80])
        for idx in df_raw.loc[today_mask].index:
            s = df_raw.loc[idx, "SUE"]
            if   s <= b[0]: df_raw.loc[idx, "SUE_Q_clean"] = 1.0
            elif s <= b[1]: df_raw.loc[idx, "SUE_Q_clean"] = 2.0
            elif s <= b[2]: df_raw.loc[idx, "SUE_Q_clean"] = 3.0
            elif s <= b[3]: df_raw.loc[idx, "SUE_Q_clean"] = 4.0
            else:           df_raw.loc[idx, "SUE_Q_clean"] = 5.0

old_q5 = (df_raw["SUE_Q"] == 5.0).sum() if "SUE_Q" in df_raw.columns else "N/A"
new_q5 = (df_raw["SUE_Q_clean"] == 5.0).sum()
print(f"\n  Old Q5 (stored, biased) : {old_q5}")
print(f"  New Q5 (expanding clean): {new_q5:,}")
if isinstance(old_q5, int):
    print(f"  Change                  : {new_q5 - old_q5:+,} signals (~{abs(new_q5-old_q5):,} affected)")

# Keep only Q5 signals
df_q5 = df_raw[df_raw["SUE_Q_clean"] == 5.0].copy()

# =============================================================================
# SECTION 3 — SIGNAL-LEVEL STATS (CAR analysis, Newey-West on CARs)
# =============================================================================
print(f"\n{SEP}")
print("  SECTION A — SIGNAL-LEVEL CAR STATISTICS")
print(SEP)

for horizon, car_col, n_days in [
        ("CAR21/R30", "CAR30", 21),
        ("CAR42/R60", "CAR60", 42),
        ("CAR63/R90", "CAR90", 63),
]:
    if car_col not in df_q5.columns:
        continue
    q5_cars  = df_q5[car_col].dropna()
    all_cars = df_raw[car_col].dropna()
    q1_cars  = df_raw.loc[df_raw["SUE_Q_clean"] == 1.0, car_col].dropna()

    n = len(q5_cars)
    mean_ = q5_cars.mean() * 100
    se    = q5_cars.std() / np.sqrt(n)

    # Newey-West t-stat on Q5 CARs (lags = sqrt(n))
    y = q5_cars.values
    X = sm.add_constant(np.ones(len(y)))
    nw = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": int(np.sqrt(len(y)))})
    nw_t = float(nw.tvalues[0])
    nw_p = float(nw.pvalues[0])

    t_q1, p_q5_q1 = stats.ttest_ind(q5_cars, q1_cars, equal_var=False)

    print(f"\n  Horizon: {n_days} trading days  [{car_col}]")
    print(f"  {'─'*58}")
    print(f"  Q5 mean CAR         : {mean_:>+.3f}%  (n={n:,})")
    print(f"  All mean CAR        : {all_cars.mean()*100:>+.3f}%")
    print(f"  Q5 vs Q1 t-stat     : t={t_q1:>+.3f}  p={p_q5_q1:.4f}")
    print(f"  NW t-stat (Q5 mean) : t={nw_t:>+.3f}  p={nw_p:.4f}  "
          f"{'✓ Significant' if nw_p < 0.05 else '✗ Not sig'}")

# =============================================================================
# SECTION 4 — LOAD PIT MEMBERSHIP
# =============================================================================
print(f"\n{SEP}")
print("  STEP 4 — LOADING POINT-IN-TIME S&P 500 MEMBERSHIP")
print(SEP)

pit_membership = {}
if os.path.exists(PIT_DB):
    conn   = sqlite3.connect(PIT_DB)
    pit_df = pd.read_sql("SELECT * FROM sp500_pit_membership", conn)
    conn.close()
    for qdate, grp in pit_df.groupby("quarter_date"):
        pit_membership[pd.Timestamp(qdate)] = set(grp["ticker"])
    print(f"  PIT quarters loaded : {len(pit_membership)}")
    print(f"  PIT range           : {min(pit_membership).date()} → {max(pit_membership).date()}")
else:
    print(f"  WARNING: {PIT_DB} not found — PIT filter disabled (survivorship bias possible)")

def pit_members_nearest(date):
    if not pit_membership:
        return None   # None = skip PIT filter
    eligible = [d for d in pit_membership if d <= date]
    return pit_membership[max(eligible)] if eligible else set()

# =============================================================================
# SECTION 5 — DOWNLOAD PRICE DATA
# =============================================================================
print(f"\n{SEP}")
print("  STEP 5 — DOWNLOADING OHLC DATA")
print(SEP)

tickers_needed = sorted(df_q5["Ticker"].unique().tolist()) + ["qqq", "QQQ", "XLI"]
print(f"  Fetching {len(tickers_needed)-3} tickers + qqq + QQQ + XLI from yfinance...")

raw_px = yf.download(
    tickers_needed,
    start="2012-06-01",
    end=str(SIM_END.date()),
    auto_adjust=True,
    progress=True,
    group_by="column",
)

px_open  = raw_px["Open"].ffill()
px_close = raw_px["Close"].ffill()

if px_open.index.tz is not None:
    px_open.index  = px_open.index.tz_localize(None)
    px_close.index = px_close.index.tz_localize(None)

all_sim_dates = pd.bdate_range(SIM_START, SIM_END)
print(f"  Prices ready for {sum(t in px_open.columns for t in tickers_needed)-3} tickers")

# =============================================================================
# SECTION 6 — FILTER SIGNALS
# =============================================================================
print(f"\n{SEP}")
print("  STEP 6 — FILTERING SIGNALS")
print(SEP)

df_filt = df_q5[df_q5["Sector"].isin(TARGET_SECTORS)].copy()
df_filt = df_filt[
    (df_filt["entry_date"] >= SIM_START) &
    (df_filt["entry_date"] <= SIM_END)
].copy()

if pit_membership:
    df_filt["in_sp500"] = df_filt.apply(
        lambda r: r["Ticker"] in pit_members_nearest(r["entry_date"]), axis=1
    )
    df_filt = df_filt[df_filt["in_sp500"]].drop(columns="in_sp500").copy()

df_filt = df_filt[df_filt["Ticker"].isin(px_open.columns)].copy()

df_filt["exit_open_date"] = df_filt["entry_date"].apply(
    lambda d: pd.bdate_range(start=d, periods=23)[-1]
)
df_filt = df_filt.sort_values("entry_date").reset_index(drop=True)

sig_by_date = {}
for _, row in df_filt.iterrows():
    sig_by_date.setdefault(row["entry_date"], []).append(row)

print(f"  Sectors               : {', '.join(sorted(TARGET_SECTORS))}")
print(f"  Signals after filters : {len(df_filt):,}")
print(f"  Signals/year avg      : {len(df_filt)/12:.0f}")
print(f"  Tickers               : {df_filt['Ticker'].nunique()}")
print(f"  Date range            : {df_filt['entry_date'].min().date()} → {df_filt['entry_date'].max().date()}")
print(f"\n  Sector breakdown:")
for sec, cnt in df_filt["Sector"].value_counts().items():
    print(f"    {sec:<30} {cnt:>5,}")

# =============================================================================
# SECTION 7 — BACKTEST ENGINE
# =============================================================================
def run_backtest(n_positions=N_POSITIONS, fee_rt=FEE_RT, capital=CAPITAL,
                 rotation_threshold=ROT_THRESH, min_hold_days=MIN_HOLD,
                 sectors=None, use_close=False):
    """
    qqq-parking walk-forward backtest.
    Entry and exit at OPEN prices (use_close=False).
    Uninvested capital is always held in qqq.
    """
    sectors = sectors or TARGET_SECTORS
    df = df_q5[df_q5["Sector"].isin(sectors)].copy()
    df = df[(df["entry_date"] >= SIM_START) & (df["entry_date"] <= SIM_END)].copy()

    if pit_membership:
        df["in_sp500"] = df.apply(
            lambda r: r["Ticker"] in pit_members_nearest(r["entry_date"]), axis=1)
        df = df[df["in_sp500"]].drop(columns="in_sp500").copy()

    df = df[df["Ticker"].isin(px_open.columns)].copy()
    df["exit_open_date"] = df["entry_date"].apply(
        lambda d: pd.bdate_range(start=d, periods=23)[-1])
    df = df.sort_values("entry_date").reset_index(drop=True)

    sbd = {}
    for _, row in df.iterrows():
        sbd.setdefault(row["entry_date"], []).append(row)

    px_entry = px_close if use_close else px_open
    px_exit  = px_close if use_close else px_open

    # Bootstrap qqq position
    qqq_init = None
    for offset in range(4):
        d = SIM_START + pd.offsets.BDay(offset)
        try:
            v = px_entry.loc[d, "qqq"]
            if pd.notna(v) and v > 0:
                qqq_init = float(v)
                break
        except KeyError:
            pass

    portfolio       = {}
    trade_log       = []
    daily_nav       = []
    total_fees_paid = 0.0
    qqq_shares      = capital / qqq_init if qqq_init else 0.0
    prev_qqq_close  = None

    for date in all_sim_dates:
        qqq_open_px  = get_px(px_entry, "qqq", date)
        qqq_close_px = get_px(px_close, "qqq", date)

        # A: Forced exits
        to_exit = [t for t, p in portfolio.items() if p["exit_open_date"] <= date]
        for tkr in to_exit:
            pos = portfolio.pop(tkr)
            ep  = get_px(px_exit, tkr, date) or get_px(px_close, tkr, date)
            if not ep or ep < pos["entry_price"] * 0.10:
                continue
            fee  = pos["shares"] * ep * fee_rt
            proc = pos["shares"] * ep * (1 - fee_rt)
            total_fees_paid += fee
            if qqq_open_px:
                qqq_shares += proc / qqq_open_px
            trade_log.append(dict(
                date=date, ticker=tkr, action="EXIT_FORCED",
                entry_price=pos["entry_price"], exit_price=ep,
                shares=pos["shares"], ret=ep/pos["entry_price"]-1,
                hold_days=(date - pos["entry_date"]).days,
                sue=pos["sue"], fee=fee,
            ))

        # B: New signals
        if date in sbd:
            for sig in sorted(sbd[date], key=lambda r: r["SUE"], reverse=True):
                tkr = sig["Ticker"]
                sue = float(sig["SUE"])
                if tkr in portfolio:
                    continue
                ep = get_px(px_entry, tkr, date)
                if ep is None or ep <= 0:
                    continue

                nav = (qqq_shares * (qqq_open_px or 1))
                for t, p in portfolio.items():
                    nav += p["shares"] * (get_px(px_entry, t, date) or p["entry_price"])

                if len(portfolio) < n_positions:
                    qqq_val   = qqq_shares * (qqq_open_px or 1)
                    slot_size = min(nav / n_positions, qqq_val / (1 + fee_rt))
                    if slot_size <= 0:
                        continue
                    fee             = slot_size * fee_rt
                    qqq_shares     -= slot_size * (1 + fee_rt) / (qqq_open_px or 1)
                    total_fees_paid += fee
                    portfolio[tkr]  = dict(
                        entry_date=date, exit_open_date=sig["exit_open_date"],
                        entry_price=ep, sue=sue,
                        shares=slot_size / ep,
                    )
                    trade_log.append(dict(
                        date=date, ticker=tkr, action="ENTER",
                        entry_price=ep, shares=slot_size/ep, sue=sue, fee=fee,
                    ))
                else:
                    min_tkr   = min(portfolio, key=lambda t: portfolio[t]["sue"])
                    min_sue   = portfolio[min_tkr]["sue"]
                    held_days = (date - portfolio[min_tkr]["entry_date"]).days

                    if sue > min_sue + rotation_threshold and held_days >= min_hold_days:
                        eject    = portfolio.pop(min_tkr)
                        eject_px = get_px(px_entry, min_tkr, date)
                        if eject_px is None or eject_px < eject["entry_price"] * 0.10:
                            portfolio[min_tkr] = eject
                            continue
                        fee_exit  = eject["shares"] * eject_px * fee_rt
                        proceeds  = eject["shares"] * eject_px * (1 - fee_rt)
                        total_fees_paid += fee_exit
                        trade_log.append(dict(
                            date=date, ticker=min_tkr, action="EXIT_ROTATED",
                            entry_price=eject["entry_price"], exit_price=eject_px,
                            shares=eject["shares"], ret=eject_px/eject["entry_price"]-1,
                            hold_days=(date - eject["entry_date"]).days,
                            sue=eject["sue"], fee=fee_exit,
                        ))
                        slot_size       = nav / n_positions
                        fee_entry       = slot_size * fee_rt
                        shares          = slot_size / (ep * (1 + fee_rt))
                        cost            = shares * ep * (1 + fee_rt)
                        total_fees_paid += fee_entry
                        if qqq_open_px:
                            qqq_shares += (proceeds - cost) / qqq_open_px
                        portfolio[tkr] = dict(
                            entry_date=date, exit_open_date=sig["exit_open_date"],
                            entry_price=ep, sue=sue, shares=shares,
                        )
                        trade_log.append(dict(
                            date=date, ticker=tkr, action="ENTER",
                            entry_price=ep, shares=shares, sue=sue, fee=fee_entry,
                        ))

        # C: Daily NAV
        qqq_px  = get_px(px_close, "qqq", date) or prev_qqq_close or 0
        nav_eod = qqq_shares * qqq_px
        n_pos   = len(portfolio)
        for tkr, pos in portfolio.items():
            px_ = get_px(px_close, tkr, date) or pos["entry_price"]
            nav_eod += pos["shares"] * (
                px_ if px_ > pos["entry_price"] * 0.10 else pos["entry_price"]
            )
        daily_nav.append(dict(date=date, nav=nav_eod, n_pos=n_pos))
        if qqq_close_px:
            prev_qqq_close = qqq_close_px

    nav_df = pd.DataFrame(daily_nav).set_index("date")
    eq     = nav_df["nav"] / capital
    fv     = eq.first_valid_index()
    eq     = eq.loc[fv:] if fv else eq
    tl_df  = pd.DataFrame(trade_log)

    return eq, nav_df, tl_df, total_fees_paid


# =============================================================================
# SECTION 8 — RUN MAIN BACKTEST
# =============================================================================
print(f"\n{SEP}")
print(f"  RUNNING MAIN BACKTEST  (N={N_POSITIONS}, fee={FEE_RT*10000:.0f}bps, "
      f"sectors={TARGET_SECTORS})")
print(SEP)

eq_base, nav_df_base, tl_base, fees_paid = run_backtest()

# Benchmark equity curves
first_date = eq_base.index[0]
eq_qq = build_bench(px_close, {"qqq": 1.0},             eq_base.index, first_date)
eq_qqq = build_bench(px_close, {"QQQ": 1.0},             eq_base.index, first_date)
eq_mix = build_bench(px_close, {"QQQ": 0.7, "XLI": 0.3}, eq_base.index, first_date)

# Daily returns
dr_s   = eq_base.pct_change(fill_method=None).dropna()
dr_qqq = eq_qqq.pct_change(fill_method=None).dropna()
dr_qqq = eq_qqq.pct_change(fill_method=None).dropna()
dr_mix = eq_mix.pct_change(fill_method=None).dropna()
dr_s, dr_qqq = dr_s.align(dr_qqq, join="inner")
_, dr_qqq    = dr_s.align(dr_qqq, join="inner")
_, dr_mix    = dr_s.align(dr_mix, join="inner")
excess_qqq = (dr_s - dr_qqq).dropna()

m_s   = perf(eq_base)
m_qqq = perf(eq_qqq)
m_qqq = perf(eq_qqq)
m_mix = perf(eq_mix)

# =============================================================================
# SECTION 9 — CORE RETURNS OUTPUT
# =============================================================================
print(f"\n{SEP}")
print(f"  STRATEGY RETURNS  |  {SIM_START.year}–{SIM_END.year}")
print(SEP)

cum_ret   = m_s["cum"]
end_val   = CAPITAL * (1 + cum_ret)
profit    = end_val - CAPITAL
qqq_end   = CAPITAL * (1 + m_qqq["cum"])
qqq_profit = qqq_end - CAPITAL
qqq_end   = CAPITAL * (1 + m_qqq["cum"])
mix_end   = CAPITAL * (1 + m_mix["cum"])
n_yrs     = len(dr_s) / 252
edge_qqq  = m_s["cagr"] - m_qqq["cagr"]
edge_qqq  = m_s["cagr"] - m_qqq["cagr"]
edge_mix  = m_s["cagr"] - m_mix["cagr"]

# DCA projection
monthly_contrib = CAPITAL / 12
fv_strat = sum(monthly_contrib * (1+m_s["cagr"])**((n_yrs*12 - i)/12)
               for i in range(int(n_yrs*12)))
fv_qqq   = sum(monthly_contrib * (1+m_qqq["cagr"])**((n_yrs*12 - i)/12)
               for i in range(int(n_yrs*12)))

print(f"\n  RETURNS")
print(f"  {SEP2}")
print(f"  Strategy CAGR         : {m_s['cagr']*100:>+.2f}%")
print(f"  qqq CAGR              : {m_qqq['cagr']*100:>+.2f}%")
print(f"  QQQ CAGR              : {m_qqq['cagr']*100:>+.2f}%")
print(f"  70/30 blend CAGR      : {m_mix['cagr']*100:>+.2f}%")
print(f"  Edge vs qqq           : {edge_qqq*100:>+.2f}pp")
print(f"  Edge vs QQQ           : {edge_qqq*100:>+.2f}pp")
print(f"  Edge vs 70/30         : {edge_mix*100:>+.2f}pp")
print(f"  Cumulative return     : {cum_ret*100:>+.2f}%")

print(f"\n  DOLLAR RETURNS  (starting capital ${CAPITAL:,})")
print(f"  {SEP2}")
print(f"  Strategy ending value : ${end_val:>12,.2f}")
print(f"  Strategy total profit : ${profit:>12,.2f}")
print(f"  qqq ending value      : ${qqq_end:>12,.2f}")
print(f"  QQQ ending value      : ${qqq_end:>12,.2f}")
print(f"  70/30 ending value    : ${mix_end:>12,.2f}")
print(f"  Dollar alpha vs qqq   : ${profit - qqq_profit:>12,.2f}")
print(f"  Total fees paid       : ${fees_paid:>12,.2f}")
print(f"  Net alpha after fees  : ${profit - qqq_profit - fees_paid:>12,.2f}")

print(f"\n  DCA  (${monthly_contrib:,.0f}/mo = ${CAPITAL:,}/yr)")
print(f"  {SEP2}")
print(f"  Strategy portfolio    : ${fv_strat:>12,.2f}")
print(f"  qqq portfolio         : ${fv_qqq:>12,.2f}")
print(f"  Extra wealth vs qqq   : ${fv_strat - fv_qqq:>12,.2f}")

# =============================================================================
# SECTION 10 — RISK METRICS
# =============================================================================
print(f"\n{SEP}")
print("  CORE RISK METRICS")
print(SEP)

print(f"\n  STRATEGY")
print(f"  {SEP2}")
print(f"  Annualised Vol        : {m_s['vol']*100:>+.2f}%")
print(f"  Sharpe Ratio          : {m_s['sharpe']:>+.3f}")
print(f"  Sortino Ratio         : {m_s['sortino']:>+.3f}")
print(f"  Calmar Ratio          : {m_s['calmar']:>+.3f}")
print(f"  Max Drawdown          : {m_s['mdd']*100:>+.2f}%")
print(f"  Skewness              : {m_s['skew']:>+.3f}")
print(f"  Excess Kurtosis       : {m_s['kurt']:>+.3f}")
print(f"  VaR 95%               : {m_s['var95']*100:>+.2f}%")
print(f"  CVaR 95% (ES)         : {m_s['cvar95']*100:>+.2f}%")
print(f"  VaR 99%               : {m_s['var99']*100:>+.2f}%")
print(f"  CVaR 99% (ES)         : {m_s['cvar99']*100:>+.2f}%")
print(f"  Omega Ratio           : {omega(eq_base):>+.3f}")
print(f"  Mean daily return     : {dr_s.mean()*100:>+.4f}%")

# Return distribution percentiles
p5_day  = np.percentile(dr_s, 5)
p1_day  = np.percentile(dr_s, 1)
print(f"\n  Daily return distribution:")
print(f"  Worst 5% of days avg  : {dr_s[dr_s <= p5_day].mean()*100:>+.2f}%")
print(f"  Worst 1% of days avg  : {dr_s[dr_s <= p1_day].mean()*100:>+.2f}%")
print(f"  Best  5% of days avg  : {dr_s[dr_s >= np.percentile(dr_s, 95)].mean()*100:>+.2f}%")

# =============================================================================
# SECTION 11 — STATISTICAL SIGNIFICANCE
# =============================================================================
print(f"\n{SEP}")
print("  STATISTICAL SIGNIFICANCE TESTS")
print(SEP)

# Test 1: One-sample t-test on daily returns
t_stat1, p_val1 = stats.ttest_1samp(dr_s.values, 0)
print(f"\n  TEST 1 — One-sample t-test (daily returns)")
print(f"  {SEP2}")
print(f"  t-statistic           : {t_stat1:>+.3f}")
print(f"  p-value               : {p_val1:.6f}")
print(f"  Mean daily return     : {dr_s.mean()*100:>+.4f}%")
print(f"  Result                : {'✓ PASS' if p_val1 < 0.05 else '✗ FAIL'}  "
      f"({'sig' if p_val1 < 0.05 else 'not sig'} @ 5%)")

# Test 2: Newey-West HAC on excess returns vs qqq
y   = excess_qqq.values
X   = sm.add_constant(np.ones(len(y)))
nw  = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 22})
nw_t = float(nw.tvalues[0])
nw_p = float(nw.pvalues[0])
print(f"\n  TEST 2 — Newey-West HAC (excess returns vs qqq, lags=22)")
print(f"  {SEP2}")
print(f"  t-statistic           : {nw_t:>+.3f}")
print(f"  p-value               : {nw_p:.4f}")
print(f"  Result                : {'✓ PASS' if nw_p < 0.05 else '✗ FAIL'}  "
      f"({'sig' if nw_p < 0.05 else 'not sig'} @ 5%)")

# Test 3: CAPM regression (HAC)
X_c    = sm.add_constant(dr_qqq.values)
capm   = sm.OLS(dr_s.values, X_c).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
alpha_d  = capm.params[0]
beta_c   = capm.params[1]
alpha_ann = alpha_d * 252
alpha_pval = capm.pvalues[0]
r2_capm    = capm.rsquared
print(f"\n  TEST 3 — CAPM Regression (HAC)")
print(f"  {SEP2}")
print(f"  Jensen's alpha (ann.) : {alpha_ann*100:>+.2f}%  (p={alpha_pval:.4f}  "
      f"{'sig' if alpha_pval < 0.05 else 'not sig'})")
print(f"  Market beta           : {beta_c:>+.3f}")
print(f"  R-squared             : {r2_capm:>.3f}")
print(f"  Unexplained return    : {(1-r2_capm)*100:.1f}%")

# Test 4: Lo (2002) Sharpe t-statistic
T        = len(dr_s)
sr_stat  = m_s["sharpe"]
sr_tstat = sr_stat * np.sqrt(T / 252)
sr_pval  = float(1 - stats.norm.cdf(sr_tstat))
sr_qqq   = m_qqq["sharpe"]
print(f"\n  TEST 4 — Sharpe significance (Lo 2002)")
print(f"  {SEP2}")
print(f"  Strategy Sharpe       : {sr_stat:>+.3f}")
print(f"  qqq Sharpe            : {sr_qqq:>+.3f}")
print(f"  t-statistic           : {sr_tstat:>+.3f}")
print(f"  p-value               : {sr_pval:.4f}")
print(f"  Result                : {'✓ PASS' if sr_pval < 0.05 else '✗ FAIL'}  "
      f"({'sig' if sr_pval < 0.05 else 'not sig'} @ 5%)")

# Test 5: Fama-French regressions (if CSV files present)
def load_ff(path):
    """Load Kenneth French daily factor CSV. Returns daily fraction returns."""
    if not os.path.exists(path):
        return None
    try:
        ff = pd.read_csv(path, skiprows=3, index_col=0)
        ff.index = pd.to_datetime(ff.index.astype(str).str.strip(), format="%Y%m%d", errors="coerce")
        ff = ff[ff.index.notna()].copy()
        ff = ff.apply(pd.to_numeric, errors="coerce") / 100
        return ff
    except Exception:
        return None

ff5  = load_ff(FF_CSV)
mom  = load_ff(MOM_CSV)

if ff5 is not None:
    for label, factors in [
        ("FF3", ["Mkt-RF","SMB","HML"]),
        ("FF5", ["Mkt-RF","SMB","HML","RMW","CMA"]),
        ("FF6 (MOM)", ["Mkt-RF","SMB","HML","RMW","CMA","Mom"]),
    ]:
        avail = [f for f in factors if f in ff5.columns or (mom is not None and f == "Mom")]
        if not avail:
            continue
        ff_use = ff5.copy()
        if "Mom" in avail and mom is not None:
            mom_col = mom.iloc[:, 0].rename("Mom")
            ff_use = ff_use.join(mom_col, how="left")

        y_ff, X_ff = dr_s.align(ff_use[avail], join="inner")
        rf_ff = ff_use["RF"].reindex(y_ff.index) if "RF" in ff_use.columns else pd.Series(0, index=y_ff.index)
        exc_ff = y_ff - rf_ff
        X_reg = sm.add_constant(X_ff)
        m_ff  = sm.OLS(exc_ff.values, X_reg.values).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
        a_ann = float(m_ff.params[0]) * 252
        a_p   = float(m_ff.pvalues[0])
        print(f"\n  TEST 5{label} — {label} Alpha")
        print(f"  {SEP2}")
        print(f"  Annualised alpha      : {a_ann*100:>+.2f}%  (p={a_p:.4f}  "
              f"{'sig' if a_p < 0.05 else 'not sig'})")
        for name, coef, pv in zip(["alpha"]+avail,
                                   m_ff.params, m_ff.pvalues):
            if name != "alpha":
                print(f"  {name:<22}: {coef:>+.4f}  (p={pv:.3f})")
else:
    print(f"\n  TEST 5 — FF Regressions")
    print(f"  {SEP2}")
    print(f"  Skipped — Fama-French CSV not found ({FF_CSV})")
    print(f"  Download from https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html")

# Information Ratio and Appraisal Ratio
ir_val = (excess_qqq.mean() / excess_qqq.std() * np.sqrt(252)
          if excess_qqq.std() != 0 else 0)
appraisal = (alpha_ann / (dr_s - beta_c*dr_qqq).std() / np.sqrt(252)
             if (dr_s - beta_c*dr_qqq).std() != 0 else 0)
print(f"\n  SUMMARY RATIOS")
print(f"  {SEP2}")
print(f"  Information Ratio     : {ir_val:>+.3f}")
print(f"  Appraisal Ratio       : {appraisal:>+.3f}")
print(f"  Treynor Ratio         : {(m_s['cagr']-RF)/beta_c:>+.4f}")

# =============================================================================
# SECTION 12 — MONTE CARLO SIMULATION
# =============================================================================
print(f"\n{SEP}")
print("  MONTE CARLO SIMULATION  (10,000 random portfolios)")
print(SEP)

np.random.seed(42)
N_MC = 10_000
boot_cagrs = []
for _ in range(N_MC):
    s = np.random.choice(dr_s.values, size=len(dr_s), replace=True)
    boot_cagrs.append(np.prod(1 + s) ** (252 / len(s)) - 1)

boot_arr  = np.array(boot_cagrs)
ci_lo, ci_hi = np.percentile(boot_arr, [2.5, 97.5])
pct_beat_qqq = (boot_arr > m_qqq["cagr"]).mean() * 100

# Bootstrap CI for Sharpe
boot_sharpes = []
for _ in range(N_MC):
    s = np.random.choice(dr_s.values, size=len(dr_s), replace=True)
    boot_sharpes.append(
        (np.mean(s) - RF/252) / np.std(s, ddof=1) * np.sqrt(252)
        if np.std(s, ddof=1) > 0 else 0
    )
sr_ci_lo, sr_ci_hi = np.percentile(boot_sharpes, [2.5, 97.5])

print(f"\n  Strategy CAGR         : {m_s['cagr']*100:>+.2f}%")
print(f"  MC mean CAGR          : {boot_arr.mean()*100:>+.2f}%")
print(f"  MC 95% CI             : [{ci_lo*100:>+.2f}%, {ci_hi*100:>+.2f}%]")
print(f"  % MC portfolios > qqq : {pct_beat_qqq:.1f}%")
print(f"  (only {pct_beat_qqq:.1f}% of random orderings beat qqq — strategy needs signal timing)")
print(f"\n  Bootstrap Sharpe 95% CI : [{sr_ci_lo:>+.3f}, {sr_ci_hi:>+.3f}]")
print(f"  Entire CI above zero    : {'Yes' if sr_ci_lo > 0 else 'No'}")
print(f"  Entire CI above qqq     : {'Yes' if sr_ci_lo > sr_qqq else 'No'}")

# =============================================================================
# SECTION 13 — OUTLIER FRAGILITY
# =============================================================================
print(f"\n{SEP}")
print("  OUTLIER FRAGILITY ANALYSIS")
print(SEP)

exits = tl_base[tl_base["ret"].notna()].copy()
n_total  = len(exits)
base_cum = float(eq_base.iloc[-1] - 1)
base_yrs = len(eq_base) / 252
qqq_cagr = m_qqq["cagr"]

print(f"\n  (Removes top X% of trades by return and re-estimates CAGR)")
print(f"  Baseline CAGR         : {m_s['cagr']*100:>+.2f}%")
print(f"  qqq benchmark         : {qqq_cagr*100:>+.2f}%")
print(f"  Total exits           : {n_total:,}")
print(f"  {'─'*58}")
print(f"  {'Threshold':<38} {'Trades rmvd':>11} {'Adj. CAGR':>10}")
print(f"  {'─'*58}")

for pct in [1, 2, 5, 10]:
    thr      = np.percentile(exits["ret"], 100 - pct)
    removed  = exits[exits["ret"] >= thr]
    n_rm     = len(removed)
    rm_sum   = removed["ret"].sum()
    adj_cum  = (1 + base_cum) * (1 - (1/N_POSITIONS) * rm_sum / n_total) - 1
    adj_cagr = (1 + adj_cum) ** (1 / base_yrs) - 1
    flag     = "✓ still above qqq" if adj_cagr > qqq_cagr else "✗ below qqq"
    print(f"  Remove top {pct:>2}% (ret >= {thr:>+.2f})   {n_rm:>6,} trades   "
          f"{adj_cagr*100:>+8.2f}%  {flag}")

# =============================================================================
# SECTION 14 — FEE SENSITIVITY
# =============================================================================
print(f"\n{SEP}")
print("  FEE & SLIPPAGE SENSITIVITY")
print(SEP)

print(f"\n  {'Fee (bps/side)':<16} {'CAGR':>8}  {'vs baseline':>12}")
print(f"  {'─'*40}")
fee_baseline_cagr = m_s["cagr"]
for bps in [5, 10, 16, 20, 30, 50, 75, 100]:
    eq_f, _, _, _ = run_backtest(fee_rt=bps/10000)
    c = cagr(eq_f)
    flag = "← baseline" if bps == 16 else ""
    print(f"  {bps:<16} {c*100:>+7.2f}%  {(c-fee_baseline_cagr)*100:>+11.2f}pp  {flag}")

# =============================================================================
# SECTION 15 — CRISIS / REGIME ANALYSIS
# =============================================================================
print(f"\n{SEP}")
print("  CRISIS & REGIME ANALYSIS")
print(SEP)

events = [
    ("Taper Tantrum 2013",     "2013-05-01", "2013-09-30"),
    ("Oil Crash 2015-16",      "2015-07-01", "2016-02-28"),
    ("Vol Spike Feb 2018",     "2018-02-01", "2018-03-31"),
    ("Q4 Selloff 2018",        "2018-10-01", "2018-12-31"),
    ("COVID Crash 2020",       "2020-02-19", "2020-03-23"),
    ("COVID Recovery 2020",    "2020-03-23", "2020-12-31"),
    ("Inflation Bear 2022",    "2022-01-01", "2022-12-31"),
    ("AI Rally 2023",          "2023-01-01", "2023-12-31"),
    ("Rate-Cut Rally 2024",    "2024-01-01", "2024-12-31"),
]

print(f"\n  {'Event':<28} {'Strategy':>10} {'qqq':>8} {'Edge':>8}")
print(f"  {'─'*58}")
for label, start, end in events:
    s_s = eq_base.loc[start:end].dropna()
    s_b = eq_qqq.loc[start:end].dropna()
    if len(s_s) < 5 or len(s_b) < 5:
        continue
    r_s = s_s.iloc[-1] / s_s.iloc[0] - 1
    r_b = s_b.iloc[-1] / s_b.iloc[0] - 1
    edge = r_s - r_b
    flag = "✓ beat" if edge > 0 else "✗ miss"
    print(f"  {label:<28} {r_s*100:>+9.2f}% {r_b*100:>+7.2f}% {edge*100:>+7.2f}%  {flag}")

# 4-era sub-period analysis
print(f"\n  SUB-PERIOD (ERA) ANALYSIS")
print(f"  {SEP2}")
eras = [
    ("2013-2015  Low vol/rates", "2013-01-01", "2015-12-31"),
    ("2016-2019  Tech bull",     "2016-01-01", "2019-12-31"),
    ("2020-2022  COVID+inflation","2020-01-01","2022-12-31"),
    ("2023-2025  AI rally",      "2023-01-01", "2025-12-31"),
]
print(f"\n  {'Era':<28} {'Strategy':>10} {'qqq':>8} {'Edge':>8} {'Sharpe':>8}")
print(f"  {'─'*62}")
era_res = []
for label, start, end in eras:
    s_s = eq_base.loc[start:end].dropna()
    s_b = eq_qqq.loc[start:end].dropna()
    if len(s_s) < 60:
        continue
    c_s  = cagr(s_s / s_s.iloc[0])
    c_b  = cagr(s_b / s_b.iloc[0])
    sh_s = sharpe(s_s / s_s.iloc[0])
    edge = c_s - c_b
    era_res.append((label, c_s, c_b, edge))
    print(f"  {label:<28} {c_s*100:>+9.2f}% {c_b*100:>+7.2f}% {edge*100:>+7.2f}pp {sh_s:>+7.3f}")

# =============================================================================
# SECTION 16 — ALPHA DECAY ANALYSIS
# =============================================================================
print(f"\n{SEP}")
print("  ALPHA DECAY ANALYSIS")
print(SEP)

# Rolling 12-month Jensen's alpha
aligned = pd.concat({"strat": dr_s, "qqq": dr_qqq}, axis=1).dropna()
window  = 252
roll_alpha_vals = []
roll_dates      = []
for i in range(window, len(aligned)+1):
    sub_s = aligned["strat"].iloc[i-window:i]
    sub_b = aligned["qqq"].iloc[i-window:i]
    Xr = sm.add_constant(sub_b.values)
    try:
        mr = sm.OLS(sub_s.values, Xr).fit()
        roll_alpha_vals.append(mr.params[0] * 252)
    except Exception:
        roll_alpha_vals.append(np.nan)
    roll_dates.append(aligned.index[i-1])

roll_alpha = pd.Series(roll_alpha_vals, index=roll_dates).dropna()
pct_positive = (roll_alpha > 0).mean() * 100

# OLS trend on rolling alpha
t_idx = np.arange(len(roll_alpha))
trend = np.polyfit(t_idx, roll_alpha.values, 1)
trend_coef_ann = trend[0] * 252   # per year
trend_coef_mo  = trend[0] * 21    # per month (approx)

# Mann-Kendall test
mk_trend, mk_s, mk_var, mk_z, mk_p = mann_kendall(roll_alpha.values)
trend_str = f"{'decreasing' if mk_z < 0 else 'increasing'}"

# Chow structural break test
break_date = pd.Timestamp("2020-01-01")
pre_  = aligned.loc[:break_date]
post_ = aligned.loc[break_date:]

def ols_rss(s, b):
    if len(s) < 5:
        return np.nan, 1
    X = sm.add_constant(b.values)
    m = sm.OLS(s.values, X).fit()
    return m.ssr, len(m.params)

rss_full, k = ols_rss(aligned["strat"], aligned["qqq"])
rss_pre,  _ = ols_rss(pre_["strat"],  pre_["qqq"])
rss_post, _ = ols_rss(post_["strat"], post_["qqq"])

if rss_full and rss_pre and rss_post:
    n1, n2 = len(pre_), len(post_)
    N      = n1 + n2
    chow_f = ((rss_full - (rss_pre + rss_post)) / k) / ((rss_pre + rss_post) / (N - 2*k))
    chow_p = float(1 - stats.f.cdf(abs(chow_f), k, N-2*k))
else:
    chow_f, chow_p = np.nan, np.nan

# Monthly excess returns
monthly_excess = excess_qqq.resample("ME").sum()
pre_excess  = monthly_excess.loc[:break_date]
post_excess = monthly_excess.loc[break_date:]
pre_mean  = pre_excess.mean()
post_mean = post_excess.mean()

t_idx_m = np.arange(len(monthly_excess))
ols_m   = np.polyfit(t_idx_m, monthly_excess.values, 1)
trend_m_coef = ols_m[0]
trend_m_ann  = trend_m_coef * 12

# OLS p-value on monthly trend
y_m  = monthly_excess.values
X_m  = sm.add_constant(np.arange(len(y_m)).reshape(-1,1))
ols_reg = sm.OLS(y_m, X_m).fit()
trend_p = float(ols_reg.pvalues[1])

# Hurst exponents
H_strat = hurst_exponent(dr_s.values)
H_qqq_  = hurst_exponent(dr_qqq.values)

# CAPM-derived annualised alpha and IR
alpha_ann_capm = alpha_ann  # computed earlier
ir_val_decay   = ir_val

print(f"\n  CAPM Jensen's alpha   : {alpha_ann_capm*100:>+.2f}%  (p={alpha_pval:.4f}  "
      f"{'sig' if alpha_pval < 0.05 else 'not sig'})")
print(f"  Information Ratio     : {ir_val_decay:>+.3f}")
print(f"  Appraisal Ratio       : {appraisal:>+.3f}")
print(f"\n  Rolling 12M alpha windows positive: {pct_positive:.1f}%")
print(f"  OLS trend (rolling α) : {trend_coef_ann*100:>+.3f}%/yr")
print(f"\n  Monthly excess OLS trend  : {trend_m_ann*100:>+.4f}%/yr  p={trend_p:.4f}  "
      f"{'sig' if trend_p < 0.05 else 'not sig'}")
print(f"\n  Mann-Kendall (rolling α)  : {trend_str}  z={mk_z:>+.3f}  p={mk_p:.4f}  "
      f"{'sig' if mk_p < 0.05 else 'not sig'}")

print(f"\n  Chow structural break @ Jan 2020:")
print(f"  F-stat                : {chow_f:.3f}")
print(f"  p-value               : {chow_p:.4f}  "
      f"{'✗ Break detected' if chow_p < 0.05 else '✓ No structural break'}")
print(f"  Pre-2020 avg monthly excess : {pre_mean*100:>+.2f}%")
print(f"  Post-2020 avg monthly excess: {post_mean*100:>+.2f}%")

print(f"\n  Hurst exponent:")
print(f"  Strategy H            : {H_strat:.4f}  "
      f"({'persistent/momentum' if H_strat > 0.55 else 'near random walk' if H_strat > 0.45 else 'mean-reverting'})")
print(f"  qqq H                 : {H_qqq_:.4f}")

# Sustainability score
checks = [
    not (trend_m_coef < 0 and trend_p < 0.05),
    not (mk_z < 0 and mk_p < 0.05),
    not (trend_coef_ann < 0 and trend_p < 0.05),
    (not np.isnan(chow_p)) and chow_p >= 0.05,
    H_strat >= 0.45,
    sum(e > 0 for _, c, b, e in era_res) >= len(era_res) // 2,
]
n_pass = sum(checks)
print(f"\n  SUSTAINABILITY SCORE  : {n_pass}/{len(checks)} checks passed")

# =============================================================================
# SECTION 17 — BENCHMARK COMPARISON
# =============================================================================
print(f"\n{SEP}")
print("  BENCHMARK COMPARISON")
print(SEP)

def risk_adj_suite(strat_dr, bench_dr, strat_sharpe, bench_sharpe, bench_vol, rf=RF):
    s, b   = strat_dr.align(bench_dr, join="inner")
    excess = (s - b).dropna()
    n      = len(s)
    rf_d   = rf / 252

    ir = excess.mean() / excess.std() * np.sqrt(252) if excess.std() != 0 else 0
    te = excess.std() * np.sqrt(252) * 100

    Xb   = sm.add_constant(b.values)
    capm_ = sm.OLS(s.values, Xb).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    a    = capm_.params[0] * 252 * 100
    bt   = capm_.params[1]
    ap   = capm_.pvalues[0]

    treynor   = ((s.mean() - rf_d) * 252) / bt if bt != 0 else 0
    strat_vol = s.std() * np.sqrt(252)
    m2        = (strat_sharpe - bench_sharpe) * bench_vol * 100

    g = s[s > 0] - 0
    l = 0 - s[s <= 0]
    om = g.sum() / l.sum() if l.sum() != 0 else np.inf

    up_b    = b[b > 0]
    dn_b    = b[b < 0]
    up_cap  = s[up_b.index].mean() / up_b.mean() if len(up_b) > 0 else 0
    dn_cap  = s[dn_b.index].mean() / dn_b.mean() if len(dn_b) > 0 else 0

    # Jobson-Korkie
    mu_s, mu_b   = s.mean(), b.mean()
    sig_s, sig_b = s.std(), b.std()
    cov_sb       = np.cov(s.values, b.values)[0,1]
    jk_var = (1/n) * (
        2*sig_s**2*sig_b**2 - 2*sig_s*sig_b*cov_sb
        + 0.5*mu_s**2*sig_b**2 + 0.5*mu_b**2*sig_s**2
        - (mu_s*mu_b*cov_sb**2)/(sig_s*sig_b + 1e-12)
    ) / (sig_s**2*sig_b**2 + 1e-12)
    jk_z   = (strat_sharpe - bench_sharpe) / np.sqrt(abs(jk_var) * 252)
    jk_p   = float(1 - stats.norm.cdf(jk_z))

    return dict(ir=ir, te=te, alpha=a, beta=bt, alpha_pval=ap,
                treynor=treynor, m2=m2, omega=om,
                up_cap=up_cap, dn_cap=dn_cap, jk_z=jk_z, jk_p=jk_p)

ra_qqq = risk_adj_suite(dr_s, dr_qqq, m_s["sharpe"], m_qqq["sharpe"], m_qqq["vol"])
ra_qqq = risk_adj_suite(dr_s, dr_qqq, m_s["sharpe"], m_qqq["sharpe"], m_qqq["vol"])
ra_mix = risk_adj_suite(dr_s, dr_mix, m_s["sharpe"], m_mix["sharpe"], m_mix["vol"])

def print_ra(label, ra, bench_cagr):
    edge = m_s["cagr"] - bench_cagr
    print(f"\n  VS {label}")
    print(f"  {SEP2}")
    print(f"  Benchmark CAGR        : {bench_cagr*100:>+.2f}%")
    print(f"  Edge (annualised)     : {edge*100:>+.2f}pp")
    print(f"  IR                    : {ra['ir']:>+.3f}  "
          f"({'strong' if ra['ir']>0.5 else 'moderate' if ra['ir']>0 else 'negative'})")
    print(f"  Tracking error        : {ra['te']:>+.2f}%")
    print(f"  Beta                  : {ra['beta']:>+.3f}")
    print(f"  Jensen alpha (HAC)    : {ra['alpha']:>+.2f}%  "
          f"(p={ra['alpha_pval']:.4f}  {'sig' if ra['alpha_pval']<0.05 else 'not sig'})")
    print(f"  Treynor ratio         : {ra['treynor']:>+.4f}")
    print(f"  M-squared             : {ra['m2']:>+.2f}%")
    print(f"  Omega ratio           : {ra['omega']:>+.3f}")
    print(f"  Upside capture        : {ra['up_cap']*100:>+.1f}%")
    print(f"  Downside capture      : {ra['dn_cap']*100:>+.1f}%")
    print(f"  JK Sharpe diff test   : z={ra['jk_z']:>+.3f}  p={ra['jk_p']:.4f}  "
          f"{'sig @ 5%' if ra['jk_p']<0.05 else 'not sig'}")

print_ra("qqq              (broad market baseline)", ra_qqq, m_qqq["cagr"])
print_ra("QQQ              (tech proxy)",            ra_qqq, m_qqq["cagr"])
print_ra("70% QQQ / 30% XLI (sector-matched blend)", ra_mix, m_mix["cagr"])

# =============================================================================
# SECTION 18 — TRADE-LEVEL STATISTICS
# =============================================================================
print(f"\n{SEP}")
print("  TRADE-LEVEL STATISTICS")
print(SEP)

exits_tl = tl_base[tl_base["ret"].notna()].copy()
win_rate = (exits_tl["ret"] > 0).mean() * 100
avg_win  = exits_tl.loc[exits_tl["ret"] > 0, "ret"].mean() * 100
avg_loss = exits_tl.loc[exits_tl["ret"] < 0, "ret"].mean() * 100
wl_ratio = abs(avg_win / avg_loss) if avg_loss != 0 else np.inf
avg_hold = exits_tl["hold_days"].mean()
exp_val  = (win_rate/100 * avg_win) + ((1 - win_rate/100) * avg_loss)

n_entries  = int((tl_base["action"] == "ENTER").sum())
n_forced   = int((tl_base["action"] == "EXIT_FORCED").sum())
n_rotated  = int((tl_base["action"] == "EXIT_ROTATED").sum())
avg_pos    = nav_df_base["n_pos"].mean()

print(f"\n  Win rate              : {win_rate:>+.1f}%")
print(f"  Avg winning trade     : {avg_win:>+.2f}%")
print(f"  Avg losing trade      : {avg_loss:>+.2f}%")
print(f"  Win/loss ratio        : {wl_ratio:>+.2f}x")
print(f"  Expected value/trade  : {exp_val:>+.2f}%")
print(f"  Avg hold period       : {avg_hold:.1f} days")
print(f"\n  ACTIVITY")
print(f"  {SEP2}")
print(f"  Total entries         : {n_entries:,}")
print(f"  Forced exits (day 22) : {n_forced:,}")
print(f"  Rotation exits        : {n_rotated:,}")
print(f"  Total fees paid       : ${fees_paid:>10,.2f}")
print(f"  Avg concurrent pos    : {avg_pos:.2f} / {N_POSITIONS}")

# =============================================================================
# SECTION 19 — YEAR-BY-YEAR
# =============================================================================
print(f"\n{SEP}")
print("  YEAR-BY-YEAR RETURNS")
print(SEP)

yoy_s   = eq_base.resample("YE").last().pct_change(fill_method=None).dropna()
yoy_qqq = eq_qqq.resample("YE").last().pct_change(fill_method=None).dropna()
yoy_qqq = eq_qqq.resample("YE").last().pct_change(fill_method=None).dropna()
yoy_mix = eq_mix.resample("YE").last().pct_change(fill_method=None).dropna()

yoy_df  = pd.concat([yoy_s, yoy_qqq], axis=1).dropna()
yoy_df.columns = ["strat", "qqq"]
yoy_df.index   = yoy_df.index.year
yoy_qqq.index  = yoy_qqq.index.year
yoy_mix.index  = yoy_mix.index.year

beat_yrs = int((yoy_df["strat"] > yoy_df["qqq"]).sum())
pos_yrs  = int((yoy_df["strat"] > 0).sum())

print(f"\n  Beat qqq in {beat_yrs}/{len(yoy_df)} calendar years")
print(f"  Positive abs. return in {pos_yrs}/{len(yoy_df)} years")
print(f"\n  {'Year':<6} {'Strategy':>10} {'qqq':>8} {'QQQ':>8} {'70/30':>8} {'vs qqq':>8} {'Flag'}")
print(f"  {'─'*58}")
for yr, row in yoy_df.iterrows():
    q  = yoy_qqq.get(yr, float("nan"))
    mx = yoy_mix.get(yr, float("nan"))
    flag = "beat" if row["strat"] > row["qqq"] else "miss"
    print(f"  {yr:<6} {row['strat']*100:>+9.2f}%"
          f" {row['qqq']*100:>+7.2f}%"
          f" {q*100:>+7.2f}%"
          f" {mx*100:>+7.2f}%"
          f" {(row['strat']-row['qqq'])*100:>+7.2f}%  {flag}")

# =============================================================================
# SECTION 20 — SECTOR TESTS
# =============================================================================
print(f"\n{SEP}")
print("  ROBUSTNESS: SECTOR REMOVAL TESTS")
print(SEP)

for secs, label in [
    ({"Technology"},  "Technology only"),
    ({"Industrials"}, "Industrials only"),
    (TARGET_SECTORS,  "Both (baseline)"),
]:
    eq_t, _, _, _ = run_backtest(sectors=secs)
    c  = cagr(eq_t)
    sh = sharpe(eq_t)
    print(f"  {label:<30} CAGR={c*100:>+.2f}%  Sharpe={sh:>+.3f}")

# =============================================================================
# SECTION 21 — POSITION COUNT GRID SEARCH
# =============================================================================
print(f"\n{SEP}")
print("  ROBUSTNESS: POSITION COUNT GRID SEARCH")
print(SEP)

print(f"\n  {'N positions':<14} {'CAGR':>8} {'Sharpe':>8} {'MDD':>8}")
print(f"  {'─'*44}")
for n in [3, 5, 7, 10]:
    eq_n, _, _, _ = run_backtest(n_positions=n)
    c  = cagr(eq_n)
    sh = sharpe(eq_n)
    md = max_drawdown(eq_n)
    flag = " ← baseline" if n == N_POSITIONS else ""
    print(f"  N={n:<12} {c*100:>+7.2f}% {sh:>+7.3f} {md*100:>+7.2f}%{flag}")

# =============================================================================
# FINAL SUMMARY
# =============================================================================
print(f"\n{SEP}")
print("  FINAL SUMMARY  —  ALL PRIMER STATS REPLICATED")
print(SEP)

all_tests_pass = [
    p_val1 < 0.05,
    nw_p   < 0.05,
    alpha_pval < 0.05,
    sr_pval < 0.05,
]
tests_passed = sum(all_tests_pass)

print(f"""
  Strategy period       : {SIM_START.date()} → {SIM_END.date()}
  ──────────────────────────────────────────────────────────
  CAGR (strategy)       : {m_s['cagr']*100:>+.2f}%
  CAGR (qqq)            : {m_qqq['cagr']*100:>+.2f}%
  Edge vs qqq           : {(m_s['cagr']-m_qqq['cagr'])*100:>+.2f}pp
  Cumulative return     : {m_s['cum']*100:>+.2f}%
  Starting $10k grew to : ${CAPITAL*(1+m_s['cum']):>12,.2f}

  Sharpe Ratio          : {m_s['sharpe']:>+.3f}
  Information Ratio     : {ir_val:>+.3f}
  Jensen's Alpha (CAPM) : {alpha_ann_capm*100:>+.2f}%  p={alpha_pval:.4f}
  Market Beta           : {beta_c:>+.3f}
  Max Drawdown          : {m_s['mdd']*100:>+.2f}%

  NW HAC significance   : t={nw_t:>+.3f}  p={nw_p:.4f}
  Lo Sharpe test        : t={sr_tstat:>+.3f}  p={sr_pval:.4f}
  One-sample t-test     : t={t_stat1:>+.3f}  p={p_val1:.6f}

  MC mean CAGR          : {boot_arr.mean()*100:>+.2f}%  (strat: {m_s['cagr']*100:>+.2f}%)
  MC beat QQQ           : {pct_beat_qqq:.1f}% of 10,000 random paths

  Chow break p (2020)   : {chow_p:.4f}  ({'break' if chow_p < 0.05 else 'no break'})
  Mann-Kendall (alpha)  : {mk_trend}  p={mk_p:.4f}
  Rolling alpha % pos   : {pct_positive:.1f}%
  Sustainability checks : {n_pass}/{len(checks)} passed

  Core tests passed     : {tests_passed}/4
""")

print(SEP)
print("  Script complete.")
print(SEP)

  STEP 1 — LOADING SIGNAL DATABASE
  Loaded rows        : 32,442
  Unique tickers     : 508
  Entry shifted +1BD : 34.8% of signals (after-close fix)
  Deduplicated       : 1,024 rows removed

  Building SUE_Q_clean (expanding window — no look-ahead)...
  This may take 60–120 seconds...


  Quarters: 100%|█████████████████████| 62/62 [00:02<00:00, 22.01it/s]



  Old Q5 (stored, biased) : 5577
  New Q5 (expanding clean): 6,702

  SECTION A — SIGNAL-LEVEL CAR STATISTICS

  Horizon: 21 trading days  [CAR30]
  ──────────────────────────────────────────────────────────
  Q5 mean CAR         : +0.770%  (n=6,681)
  All mean CAR        : +0.387%
  Q5 vs Q1 t-stat     : t=+3.359  p=0.0008
  NW t-stat (Q5 mean) : t=+4.407  p=0.0000  ✓ Significant

  Horizon: 42 trading days  [CAR60]
  ──────────────────────────────────────────────────────────
  Q5 mean CAR         : +1.023%  (n=6,625)
  All mean CAR        : +0.333%
  Q5 vs Q1 t-stat     : t=+3.888  p=0.0001
  NW t-stat (Q5 mean) : t=+4.147  p=0.0000  ✓ Significant

  Horizon: 63 trading days  [CAR90]
  ──────────────────────────────────────────────────────────
  Q5 mean CAR         : +1.715%  (n=6,567)
  All mean CAR        : +0.605%
  Q5 vs Q1 t-stat     : t=+5.059  p=0.0000
  NW t-stat (Q5 mean) : t=+5.451  p=0.0000  ✓ Significant

  STEP 4 — LOADING POINT-IN-TIME S&P 500 MEMBERSHIP
  PIT quarters

[*********************100%***********************]  485 of 485 completed

485 Failed downloads:
['WRB', 'GWW', 'OKE', 'ANET', 'ADI', 'DRI', 'CVNA', 'IP', 'AIG', 'NVR', 'CHD', 'TEL', 'NTRS', 'ADBE', 'AVGO', 'PRU', 'FTNT', 'BRK-B', 'CTAS', 'RJF', 'GILD', 'JCI', 'TRMB', 'HPQ', 'DD', 'CCI', 'CSGP', 'DGX', 'LHX', 'CMS', 'MGM', 'DOV', 'MO', 'CAH', 'PGR', 'HAS', 'PNW', 'MCHP', 'VZ', 'MU', 'MA', 'ESS', 'TGT', 'AFL', 'FAST', 'KMB', 'EXPE', 'MTB', 'KLAC', 'FFIV', 'AAPL', 'PHM', 'MNST', 'JBL', 'MOS', 'PLTR', 'MCK', 'PH', 'AWK', 'TSLA', 'AMAT', 'XYZ', 'EG', 'GM', 'FE', 'TYL', 'HD', 'RVTY', 'NOC', 'PG', 'IEX', 'ZTS', 'CHRW', 'SPY', 'FSLR', 'NSC', 'CTVA', 'MRNA', 'CARR', 'ROP', 'YUM', 'LIN', 'PYPL', 'DXCM', 'PEG', 'WAB', 'IVZ', 'ABT', 'INVH', 'MKC', 'LUV', 'AXON', 'FISV', 'DELL', 'DAL', 'DLTR', 'PKG', 'XLI', 'XEL', 'GOOGL', 'CCL', 'TT', 'TRGP', 'JKHY', 'GS', 'ATO', 'WM', 'GD', 'QCOM', 'TDY', 'TPL', 'ZBRA', 'ERIE', 'ABNB', 'BLK', 'CMCSA', 'EBAY', 'CRWD', 'STLD', 'ZBH', 'BG', 'LLY', 'TDG', 'NXPI', 'BE

  Prices ready for 482 tickers

  STEP 6 — FILTERING SIGNALS
  Sectors               : Industrials, Technology
  Signals after filters : 1,523
  Signals/year avg      : 127
  Tickers               : 141
  Date range            : 2013-01-17 → 2025-12-18

  Sector breakdown:
    Technology                     1,053
    Industrials                      470

  RUNNING MAIN BACKTEST  (N=7, fee=16bps, sectors={'Industrials', 'Technology'})


NameError: name 'eq_spy' is not defined